# 01 - Exploratory Data Analysis

Explore the raw data sources: KenPom/Barttorvik stats, tournament matchups, and supplemental files.

In [ ]:

import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_loader import (
    load_kenpom_barttorvik, load_tournament_matchups,
    load_resumes, load_shooting_splits
)


In [ ]:

kb = load_kenpom_barttorvik()
print(f"KenPom/Barttorvik shape: {kb.shape}")
print(f"Years: {kb['YEAR'].min()} - {kb['YEAR'].max()}")
kb.head(3)


In [ ]:

# distribution of KADJ EM across tournament teams only
tourney = kb[kb['SEED'].notna() & (kb['SEED'] > 0)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(tourney['KADJ EM'].dropna(), bins=30, edgecolor='black')
axes[0].set_title('KenPom AdjEM - Tournament Teams')
axes[0].set_xlabel('AdjEM')

axes[1].scatter(tourney['KADJ EM'], tourney['KADJ T'], alpha=0.3, s=10)
axes[1].set_xlabel('AdjEM')
axes[1].set_ylabel('AdjT (Tempo)')
axes[1].set_title('AdjEM vs Tempo')

axes[2].hist(tourney['SEED'].dropna(), bins=16, edgecolor='black')
axes[2].set_title('Seed Distribution')
axes[2].set_xlabel('Seed')

plt.tight_layout()
plt.show()


In [ ]:

tm = load_tournament_matchups()
print(f"Tournament matchups shape: {tm.shape}")
print(f"Years: {tm['YEAR'].min()} - {tm['YEAR'].max()}")
print(f"Rounds: {sorted(tm['ROUND'].unique())}")

# win rate by seed
winners = tm[tm['ROUND'] < tm['CURRENT ROUND']].copy()
losers = tm[tm['ROUND'] == tm['CURRENT ROUND']].copy()
print(f"\nWinners: {len(winners)}, Losers: {len(losers)}")


In [ ]:

# upset frequency by seed matchup
import itertools

seed_wins = {}
for _, row in tm.iterrows():
    seed = int(row['SEED'])
    won = int(row['ROUND']) < int(row['CURRENT ROUND'])
    seed_wins.setdefault(seed, []).append(int(won))

seed_winrate = {s: (sum(v) / len(v)) for s, v in seed_wins.items() if len(v) >= 5}
seeds = sorted(seed_winrate.keys())
winrates = [seed_winrate[s] for s in seeds]

plt.figure(figsize=(10, 4))
plt.bar(seeds, winrates)
plt.axhline(0.5, color='red', linestyle='--', label='50%')
plt.xlabel('Seed')
plt.ylabel('Win Rate')
plt.title('Tournament Win Rate by Seed (all rounds, 2008-present)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# correlation heatmap of key efficiency metrics
import matplotlib.pyplot as plt
import numpy as np

key_cols = ['KADJ EM', 'KADJ O', 'KADJ D', 'KADJ T', 'BARTHAG',
            'EFG%', 'EFG%D', 'TOV%', 'OREB%', 'FT%', 'PPPO', 'PPPD']
available = [c for c in key_cols if c in kb.columns]
corr = kb[available].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(len(available)))
ax.set_yticks(range(len(available)))
ax.set_xticklabels(available, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(available, fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Correlation Heatmap - Key Metrics')
plt.tight_layout()
plt.show()


In [ ]:

# show average KenPom AdjEM by round reached
round_adjem = kb[kb['SEED'].notna() & (kb['SEED'] > 0)].groupby('ROUND')['KADJ EM'].mean().sort_index()
round_labels = {64: 'R64', 32: 'R32', 16: 'S16', 8: 'E8', 4: 'F4', 2: 'Final', 1: 'Champ'}
round_adjem.index = [round_labels.get(r, str(r)) for r in round_adjem.index]

plt.figure(figsize=(8, 4))
plt.bar(round_adjem.index, round_adjem.values)
plt.xlabel('Deepest Round Reached')
plt.ylabel('Avg KenPom AdjEM')
plt.title('Average AdjEM by Deepest Tournament Round (2008-present)')
plt.tight_layout()
plt.show()
